<a href="https://colab.research.google.com/github/Harmain-Kanwal/Flyrank-AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Growing Content is Longer and Younger
The paper finds that growing content is generally longer and newer than declining content and suggests that expanding thin pages may improve growth potential.

Methodology Question

Since this is an observational comparison, I would ask whether word count actually causes growth. Longer articles may also receive more updates, target different keywords, or belong to higher-priority topics. To support a stronger claim, I would like to see whether these other variables were controlled during the analysis.

Finding 2: Click Capture by Position Tier
The paper reports that pages ranking in higher search positions achieve substantially higher weighted click-through rates.

Methodology Question

The relationship between ranking position and CTR is expected, but I would ask whether search intent and branded queries were considered. Different types of searches naturally have different click behaviour, so separating these categories could provide a more accurate interpretation of the results.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Clone the repository if running in Google Colab
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the starter dataset from the repository
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [18]:
from sklearn.model_selection import GroupShuffleSplit

df["is_declining"] = (
    df["trend_direction"].astype(str).str.lower().str.contains("declin").astype(int)
)
print(df["is_declining"].value_counts(normalize=True))

ID_COLS      = ["content_id", "client_id"]
TARGET_COLS  = ["trend_direction", "trend_pct", "is_declining"]
LEAKAGE_COLS = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]

CATEGORICAL_COLS = ["content_type", "main_intent", "provider_used", "model_used", "competition_level"]
NUMERIC_COLS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

df_encoded = pd.get_dummies(df, columns=CATEGORICAL_COLS)
dummy_cols = [c for c in df_encoded.columns if any(c.startswith(cat + "_") for cat in CATEGORICAL_COLS)]
FEATURE_COLS = NUMERIC_COLS + dummy_cols

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df_encoded, groups=df_encoded["client_id"]))

train_df = df_encoded.iloc[train_idx].reset_index(drop=True)
test_df  = df_encoded.iloc[test_idx].reset_index(drop=True)

assert set(train_df["client_id"]) & set(test_df["client_id"]) == set()
print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients, "
      f"{train_df['is_declining'].mean():.1%} declining")
print(f"Test:  {len(test_df)} rows, {test_df['client_id'].nunique()} clients, "
      f"{test_df['is_declining'].mean():.1%} declining")

is_declining
0    1.0
Name: proportion, dtype: float64
Train: 23837 rows, 25 clients, 0.0% declining
Test:  6163 rows, 7 clients, 0.0% declining


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score

# ==========================================
# Features and Target (same as Week 5)
# ==========================================

X = df_encoded[FEATURE_COLS]
y = df_encoded["is_declining"]

# ==========================================
# BEFORE: Random Train/Test Split
# ==========================================

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

dt_random = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

dt_random.fit(X_train_random, y_train_random)

random_pred = dt_random.predict(X_test_random)

random_accuracy = accuracy_score(y_test_random, random_pred)

# ==========================================
# AFTER: Honest Group Split (by client_id)
# ==========================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df_encoded,
        groups=df_encoded["client_id"]
    )
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

dt_group = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

dt_group.fit(X_train_group, y_train_group)

group_pred = dt_group.predict(X_test_group)

group_accuracy = accuracy_score(y_test_group, group_pred)

# ==========================================
# Results
# ==========================================

print("Decision Tree Performance Comparison")
print("------------------------------------")
print(f"Random Split Accuracy : {random_accuracy:.4f}")
print(f"Group Split Accuracy  : {group_accuracy:.4f}")
print(f"Difference            : {random_accuracy-group_accuracy:.4f}")

Decision Tree Performance Comparison
------------------------------------
Random Split Accuracy : 1.0000
Group Split Accuracy  : 1.0000
Difference            : 0.0000


To evaluate whether my Week 5 model was affected by data leakage, I trained the same Decision Tree classifier using two different evaluation strategies.

**Before (Random Split):**
The dataset was divided randomly into training (80%) and testing (20%) sets using stratified sampling. This approach may place records from the same client in both training and testing data, producing overly optimistic results.

**After (Honest Group Split):**
The model was re-evaluated using a GroupShuffleSplit based on `client_id`. This ensures that all records from a client appear in only one split, preventing information leakage between training and testing.

| Split Method | Accuracy |
|--------------|----------|
| Random Split | 1.0000 |
| Group Split | 1.0000 |

The group-based evaluation provides a more realistic estimate of model performance because it tests the model on completely unseen clients. Any decrease in accuracy compared with the random split indicates that the original evaluation benefited from information leakage.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Verify Dataset Grain

In [16]:
grain = df.groupby(["content_id","client_id"]).size()

print(grain.value_counts())

1    30000
Name: count, dtype: int64


### Leakage Audit

In [20]:
leakage_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print(df[leakage_cols].head())

   impressions_last_30d  clicks_last_30d  sessions_last_30d  \
0                   578                2                  2   
1                  2501                2                  3   
2                  2382                1                  1   
3                  3626               22                 35   
4                  4211               10                 14   

   impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  
0                   987               13                  9  
1                  5915                1                  2  
2                  6089                3                  3  
3                  4206               17                 26  
4                  6452                2                  9  


### Client Leakage Check

In [21]:
train_clients = set(train_df.client_id)

test_clients = set(test_df.client_id)

print("Shared clients:", len(train_clients & test_clients))

Shared clients: 0


### Leakage Audit Summary

The Week 5 feature set was audited using the same approach as Week 3.

The audit confirmed that:

- the modelling grain is consistent,
- identifier columns were excluded from training,
- future performance variables were removed,
- training and testing clients are completely separated,
- only historical information is available to the model at prediction time.

These checks reduce the risk of data leakage and provide a more reliable estimate of model performance.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim
"The Decision Tree successfully predicts declining content with perfect accuracy."

In this analysis, the Decision Tree achieved the measured performance shown on the evaluation dataset. This should be interpreted as an observed result for the available data, not evidence that the model will always perform perfectly on unseen data. The model provides directional insights that can support content refresh decisions alongside human judgment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.